# Cross-Validation in Machine Learning
## A step-by-step practical notebook

This notebook explains cross-validation with clear comments and runnable examples.

**What you will learn**
1. Why cross-validation is useful.
2. How K-Fold Cross-Validation works.
3. How to use `cross_val_score()`.
4. How Stratified K-Fold works for classification.
5. How to use a `Pipeline` to avoid data leakage during preprocessing.
6. How to use cross-validation with `GridSearchCV`.

> Run the cells from top to bottom. The examples use the built-in Iris dataset, so no external dataset download is required.

## 1. Import the required libraries

We use:
- NumPy to calculate the mean and standard deviation.
- Matplotlib to visualize the fold scores.
- Scikit-learn for datasets, models, splitting strategies, pipelines, and evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Display NumPy arrays with a manageable number of decimal places
np.set_printoptions(precision=3, suppress=True)

## 2. Load a dataset

The Iris dataset contains measurements of flowers.

- `X` contains the input features.
- `y` contains the target class.
- This is a classification problem with three classes.

In [ ]:
# Load the built-in Iris dataset
iris = load_iris()

# Input features: sepal length, sepal width, petal length, petal width
X = iris.data

# Target labels: 0, 1, or 2
y = iris.target

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Feature names:", iris.feature_names)
print("Class names:", iris.target_names)

# Count the number of samples belonging to each class
unique_classes, class_counts = np.unique(y, return_counts=True)
print("\nClass distribution:")
for class_id, count in zip(unique_classes, class_counts):
    print(f"{iris.target_names[class_id]} (class {class_id}): {count}")

## 3. Understand a normal train-test split

A normal train-test split divides the dataset once.

For example, with `test_size=0.2`:
- 80% of records are used for training.
- 20% of records are kept for testing.

The model is evaluated on just that one test split. The result can depend on which records happen to be selected.

In [ ]:
# Split the data once: 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,       # 20% of the data is reserved for testing
    random_state=42,     # Makes the split reproducible
    stratify=y            # Preserves class proportions in both sets
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

# Create and train a Logistic Regression classifier
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# Predict the classes of the test records
y_pred = model.predict(X_test)

# Calculate accuracy on this one test split
test_accuracy = accuracy_score(y_test, y_pred)
print("Single train-test split accuracy:", test_accuracy)

## 4. What is K-Fold Cross-Validation?

K-Fold divides the dataset into `K` parts, called **folds**.

For 5-fold cross-validation:
1. Divide the data into 5 folds.
2. Use fold 1 for validation and folds 2–5 for training.
3. Repeat, using fold 2 for validation.
4. Continue until every fold has been used for validation once.
5. Calculate the average of the 5 validation scores.

Each record is used for validation once and for training in the other four iterations.

**Important:** In each iteration, a fresh copy of the estimator is fitted. The same fitted model is not simply reused.

In [ ]:
# Create a 5-fold splitting strategy
kf = KFold(
    n_splits=5,       # Divide the dataset into 5 folds
    shuffle=True,     # Shuffle records before creating folds
    random_state=42   # Makes the fold assignment reproducible
)

# Show which record indices are used for training and validation in each fold
for fold_number, (train_indices, validation_indices) in enumerate(kf.split(X), start=1):
    print(f"\nFold {fold_number}")
    print("Training record count:", len(train_indices))
    print("Validation record count:", len(validation_indices))
    print("First few training indices:", train_indices[:5])
    print("First few validation indices:", validation_indices[:5])

## 5. Perform K-Fold Cross-Validation with `cross_val_score()`

`cross_val_score()` performs the repeated training and validation automatically.

Important arguments:
- `estimator`: the machine learning model.
- `X`: input features.
- `y`: target labels.
- `cv`: the cross-validation splitting strategy.
- `scoring`: the evaluation metric.

It returns one score for each fold.

In [ ]:
# Create the model
logistic_model = LogisticRegression(max_iter=500)

# Perform 5-fold cross-validation
fold_scores = cross_val_score(
    estimator=logistic_model,
    X=X,
    y=y,
    cv=kf,
    scoring="accuracy"
)

# Print the accuracy obtained in each fold
for fold_number, score in enumerate(fold_scores, start=1):
    print(f"Fold {fold_number} accuracy: {score:.4f}")

# Mean accuracy gives the average performance across all folds
mean_accuracy = np.mean(fold_scores)

# Standard deviation indicates how much the fold scores vary
std_accuracy = np.std(fold_scores)

print("\nMean cross-validation accuracy:", round(mean_accuracy, 4))
print("Standard deviation:", round(std_accuracy, 4))

In [ ]:
# Visualize the accuracy obtained in each fold
plt.figure(figsize=(7, 4))

fold_numbers = np.arange(1, len(fold_scores) + 1)
plt.bar(fold_numbers, fold_scores)

plt.xlabel("Fold number")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross-Validation Scores")
plt.xticks(fold_numbers)
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.3)

plt.show()

## 6. Stratified K-Fold Cross-Validation

For classification, especially when classes are imbalanced, **StratifiedKFold** is often preferable.

It tries to preserve approximately the same class proportions in every fold.

Example: if the full dataset contains 90% class 0 and 10% class 1, each fold will aim to contain approximately that same proportion.

- `n_splits=5`: create 5 folds.
- `shuffle=True`: shuffle before splitting.
- `random_state=42`: make the split reproducible.

Use `StratifiedKFold` for ordinary classification datasets where preserving class proportions is important.

In [ ]:
# Create a stratified 5-fold splitter
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Check the class distribution in each validation fold
for fold_number, (train_indices, validation_indices) in enumerate(skf.split(X, y), start=1):
    y_validation = y[validation_indices]

    classes, counts = np.unique(y_validation, return_counts=True)

    print(f"\nFold {fold_number} validation class distribution:")
    for class_id, count in zip(classes, counts):
        print(f"Class {class_id} ({iris.target_names[class_id]}): {count} records")

# Evaluate Logistic Regression using Stratified K-Fold
stratified_scores = cross_val_score(
    estimator=LogisticRegression(max_iter=500),
    X=X,
    y=y,
    cv=skf,
    scoring="accuracy"
)

print("\nStratified fold scores:", stratified_scores)
print("Mean stratified CV accuracy:", stratified_scores.mean())

## 7. Cross-validation with preprocessing: avoid data leakage

Suppose you want to scale features using `StandardScaler`.

**Incorrect approach:** fit the scaler on the complete dataset before cross-validation. This allows information from validation folds to influence preprocessing.

**Correct approach:** put the scaler and model inside a Scikit-learn `Pipeline`.

During each fold, the pipeline:
1. Fits the scaler only on that fold's training records.
2. Transforms that fold's training records.
3. Fits the model on the transformed training records.
4. Transforms the validation records using the training-fold scaler.
5. Evaluates the model on the validation records.

This prevents leakage from validation folds into preprocessing.

In [ ]:
# Build a pipeline: scaling happens before Logistic Regression
pipeline = Pipeline([
    ("scaler", StandardScaler()),  # Learns mean and standard deviation from training fold only
    ("model", LogisticRegression(max_iter=500))  # Classification model
])

# Cross-validation fits the complete pipeline separately in every fold
pipeline_scores = cross_val_score(
    estimator=pipeline,
    X=X,
    y=y,
    cv=skf,
    scoring="accuracy"
)

print("Scores with scaling inside the pipeline:", pipeline_scores)
print("Mean accuracy:", pipeline_scores.mean())
print("Standard deviation:", pipeline_scores.std())

## 8. Use cross-validation for hyperparameter tuning with `GridSearchCV`

Cross-validation can also help select hyperparameters.

Here, we compare different values of Logistic Regression's `C` parameter.

- `C` controls the strength of regularization.
- Smaller `C` means stronger regularization.
- Larger `C` means weaker regularization.
- `param_grid` contains the values to test.
- `GridSearchCV` evaluates each parameter setting using cross-validation and selects the best mean score.

The pipeline ensures that scaling is fitted separately inside each fold.

In [ ]:
# Define the pipeline
pipeline_for_search = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Define the hyperparameter values to test
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# GridSearchCV tests each C value using 5-fold stratified cross-validation
grid_search = GridSearchCV(
    estimator=pipeline_for_search,
    param_grid=param_grid,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,       # Use available CPU cores
    return_train_score=True
)

# Run the search on the training data
# In a real final evaluation, keep a separate test set untouched until this step is complete.
grid_search.fit(X_train, y_train)

print("Best hyperparameters:", grid_search.best_params_)
print("Best mean CV accuracy:", grid_search.best_score_)

# Show the mean validation score for every tested C value
for params, mean_score, std_score in zip(
    grid_search.cv_results_["params"],
    grid_search.cv_results_["mean_test_score"],
    grid_search.cv_results_["std_test_score"]
):
    print(params, "Mean CV accuracy:", round(mean_score, 4),
          "Std:", round(std_score, 4))

## 9. Final evaluation on an untouched test set

A good workflow is:
1. Split the original data into training and final test sets.
2. Use cross-validation only on the training set to compare models or tune hyperparameters.
3. Select the best model.
4. Evaluate it once on the untouched test set.

The final test set gives an estimate of performance on data that was not used for model selection.

In [ ]:
# Use the best pipeline found by GridSearchCV
best_model = grid_search.best_estimator_

# Predict on the untouched test set
final_predictions = best_model.predict(X_test)

# Evaluate final test performance
final_accuracy = accuracy_score(y_test, final_predictions)

print("Final test accuracy:", round(final_accuracy, 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nClassification report:")
print(classification_report(y_test, final_predictions, target_names=iris.target_names))

## 10. Important points to remember

| Concept | Meaning |
|---|---|
| Fold | One part of the dataset |
| K-Fold | Divides data into K folds and rotates the validation fold |
| `n_splits` | Number of folds |
| `shuffle` | Whether to shuffle before splitting |
| `random_state` | Makes random splitting reproducible |
| `cross_val_score()` | Returns one evaluation score per fold |
| `StratifiedKFold` | Preserves class proportions approximately |
| `Pipeline` | Keeps preprocessing inside each fold and helps prevent leakage |
| `GridSearchCV` | Uses cross-validation to select hyperparameters |

### Which method should you choose?

- **Classification:** commonly use `StratifiedKFold`.
- **Regression:** commonly use `KFold`.
- **Time-series data:** use a time-aware splitter such as `TimeSeriesSplit`; do not randomly shuffle future and past observations.
- **Preprocessing:** use a `Pipeline` so transformations are fitted only on each training fold.

### One important warning

Cross-validation scores are estimates, not a guarantee of future performance. Keep a final test set separate when you need an unbiased final evaluation. Choose a suitable scoring metric for the problem; accuracy is not always appropriate for imbalanced classification.